In [22]:
import polars as pl
def get_test_dataset():
    '''Test dataset with multiple informative singletons, pairs, and triplets.

    Effective strains: ST-1..ST-6 (ST-empty all-zero, ST-ref ref-like, both dropped).

    Informative singletons (2): A1, A2 — absent from all effective strains
    Informative pairs (2):
      (P1, P2):  P1={1,2,3}, P2={4,5,6}
      (P3, P4):  P3={1,4,5}, P4={2,3,6}
    Informative triplets (2):
      (T1, T2, T3): T1={1,2,4,5}, T2={1,3,4,6}, T3={2,3,5,6}
      (U1, U2, U3): U1={1,2,4,6}, U2={1,3,5,6}, U3={2,3,4,5}
    '''
    strain_rows = [
        # kmer   ST-1 ST-2 ST-3 ST-4 ST-5 ST-6 ST-empty ST-ref
        ["A1",    0,   0,   0,   0,   0,   0,    0,       ],
        ["A2",    0,   0,   0,   0,   0,   0,    0,       ],
        ["T1",    1,   1,   0,   1,   1,   0,    0,       ],
        ["T2",    1,   0,   1,   1,   0,   1,    0,       ],
        ["T3",    0,   1,   1,   0,   1,   1,    0,       ],
        ["U1",    1,   1,   0,   1,   0,   1,    0,       ],
        ["U2",    1,   0,   1,   0,   1,   1,    0,       ],
        ["U3",    0,   1,   1,   1,   1,   0,    0,       ],
        ["P1",    1,   1,   1,   0,   0,   0,    0,       ],
        ["P2",    0,   0,   0,   1,   1,   1,    0,       ],
        ["P3",    1,   0,   0,   1,   1,   0,    0,       ],
        ["P4",    0,   1,   1,   0,   0,   1,    0,       ],
    ]
    df = pl.DataFrame(
        strain_rows,
        schema=["#kmer", "ST-1", "ST-2", "ST-3", "ST-4", "ST-5", "ST-6", "ST-empty"],
        orient="row",
    )

    # Sample panel — design counts to exercise coverage logic.
    # Make sure to give pair partners and triplet members co-occurring counts in some samples.
    sample_rows = [
        # kmer    S1   S2   S3   S4   S5
        ["A1",    100,  50,   0,   0,   0],   # singleton observed in S1, S2
        ["A2",     20,   0,   0,   0,  10],
        ["T1",      0,   0,  50,   0,  50],   # T-triplet observed in S3 (50,40,30) and S5 (10,20,30)
        ["T2",      0,   0,  100,   0,  100],
        ["T3",      0,   0,  100,   0,  100],
        ["U1",      0,   0,   0,  50,  50],   # U-triplet observed in S4 and S5
        ["U2",      0,   0,   0,  50,  50],
        ["U3",      0,   0,   0,  50,  50],
        ["P1",      0, 100,   0,   0,   0],   # (P1,P2) pair observed in S2 (both>0)? need P2>0 too
        ["P2",      0,  50,   0,   0,   0],
        ["P3",      0,   0,   0,  50,   0],   # (P3,P4) pair observed in S4
        ["P4",      0,   0,   0,  50,   0],
    ]
    df_samples = pl.DataFrame(
        sample_rows,
        schema=["#kmer", "S1", "S2", "S3", "S4", "S5"],
        orient="row",
    )

    print("Creating Test Data")
    print("Strain panel:")
    print(df)
    
    df_long = (
        df
        .unpivot(
            index='#kmer',
            variable_name='sample_id',
            value_name='count',
        )
        .filter(pl.col('count') > 0)
        .select(['sample_id', '#kmer', 'count'])   # column order you asked for
    )
    print(df_long)
    print("Sample panel:")
    print(df_samples)
    return df, df_samples, df_long
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
from itertools import combinations
import os
from collections import defaultdict


def build_signatures_from_long(df_long, kmer_col="#kmer", strain_col="sample_id"):
    """From long-format presence data, produce signature equivalence classes.

    Returns:
      sig_df: Polars DataFrame with columns [sig_id, n_kmers, strain_set (list[int]), n_strains]
      kmer_to_sig: Polars DataFrame [#kmer, sig_id]
      strain_index: dict mapping strain name -> int (for strain_set encoding)
    """
    strains = df_long[strain_col].unique().sort().to_list()
    strain_index = {s: i for i, s in enumerate(strains)}

    # one row per kmer with its sorted list of strain indices = canonical signature
    kmer_sigs = (
        df_long
        .with_columns(pl.col(strain_col).replace_strict(strain_index).alias("_sidx"))
        .group_by(kmer_col)
        .agg(pl.col("_sidx").sort().alias("strain_set"))
        .with_columns(pl.col("strain_set").list.len().alias("n_strains"))
    )

    # group kmers by identical signatures
    sig_df = (
        kmer_sigs
        .group_by("strain_set")
        .agg([
            pl.col(kmer_col).alias("kmers"),
            pl.col(kmer_col).len().alias("n_kmers"),
            pl.col("n_strains").first(),
        ])
        .with_row_index("sig_id")
    )

    kmer_to_sig = (
        sig_df.explode("kmers")
              .select(pl.col("kmers").alias(kmer_col), "sig_id")
    )

    print(f"  {kmer_sigs.height:,} k-mers → {sig_df.height:,} unique signatures "
          f"({kmer_sigs.height / sig_df.height:.1f}x compression)")
    return sig_df, kmer_to_sig, strain_index

_PAIR_SCHEMA = pa.schema([
("kmerA", pa.string()),
("kmerB", pa.string()),
("sig_a", pa.uint32()),
("sig_b", pa.uint32()),
])


def find_informative_pairs(sig_df, output_dir, basename, batch_size=1_000_000):
    """Informative pair = two signatures with disjoint strain sets.
    Expand each informative signature pair into all kmer × kmer products.
    """
    sigs = sig_df.sort("sig_id")
    sig_sets = [frozenset(s) for s in sigs["strain_set"].to_list()]
    sig_kmers = sigs["kmers"].to_list()
    sig_ids = sigs["sig_id"].to_list()
    n_sig = len(sig_sets)

    inform_path = os.path.join(output_dir, f"{basename}.inform_kmer_pairs.parquet")
    writer = pq.ParquetWriter(inform_path, _PAIR_SCHEMA, compression="zstd")

    a_buf, b_buf, sa_buf, sb_buf = [], [], [], []
    n_inform = 0

    def flush():
        nonlocal a_buf, b_buf, sa_buf, sb_buf
        if not a_buf:
            return
        writer.write_table(pa.table(
            {"kmerA": a_buf, "kmerB": b_buf, "sig_a": sa_buf, "sig_b": sb_buf},
            schema=_PAIR_SCHEMA,
        ))
        a_buf, b_buf, sa_buf, sb_buf = [], [], [], []

    print(f"  Scanning {n_sig*(n_sig-1)//2:,} signature pairs", flush=True)
    for i, j in combinations(range(n_sig), 2):
        if sig_sets[i].isdisjoint(sig_sets[j]):
            # expand to all kmer pairs in this signature × signature block
            ka_list, kb_list = sig_kmers[i], sig_kmers[j]
            sid_a, sid_b = sig_ids[i], sig_ids[j]
            for ka in ka_list:
                for kb in kb_list:
                    a, b = (ka, kb) if ka < kb else (kb, ka)
                    a_buf.append(a); b_buf.append(b)
                    sa_buf.append(sid_a); sb_buf.append(sid_b)
                    n_inform += 1
                    if len(a_buf) >= batch_size:
                        flush()

    flush()
    writer.close()
    print(f"  Informative pairs: {n_inform:,}", flush=True)
    return n_inform
_TRIP_SCHEMA = pa.schema([
    ("kmerA", pa.string()), ("kmerB", pa.string()), ("kmerC", pa.string()),
    ("sig_a", pa.uint32()), ("sig_b", pa.uint32()), ("sig_c", pa.uint32()),
])


def find_informative_triplets(
    sig_df, output_dir, basename,
    used_kmers=None,                 # set of k-mers already in informative pairs
    batch_size=1_000_000,
    max_kmers_per_sig=None,
):
    """Informative triplet = three signatures whose 3-way intersection is empty.
    
    used_kmers: optional set of k-mers to exclude (e.g. those already in pairs).
                Signatures left with zero remaining k-mers are dropped.
    """
    sigs = sig_df.sort("sig_id").to_dicts()
    
    # filter out used k-mers; drop signatures that empty out
    pruned = []
    for row in sigs:
        kept = row["kmers"] if used_kmers is None else [k for k in row["kmers"] if k not in used_kmers]
        if kept:
            if max_kmers_per_sig is not None:
                kept = kept[:max_kmers_per_sig]
            pruned.append({
                "sig_id": row["sig_id"],
                "strain_set": frozenset(row["strain_set"]),
                "kmers": kept,
            })
    
    n_sig = len(pruned)
    print(f"  Triplet search over {n_sig} signatures (after dropping pair-used k-mers)", flush=True)
    if n_sig < 3:
        print("  Fewer than 3 signatures remaining — no triplets possible.", flush=True)
        return 0

    inform_path = os.path.join(output_dir, f"{basename}.inform_kmer_triplets.parquet")
    writer = pq.ParquetWriter(inform_path, _TRIP_SCHEMA, compression="zstd")

    a_buf, b_buf, c_buf = [], [], []
    sa_buf, sb_buf, sc_buf = [], [], []
    n_inform = 0

    def flush():
        nonlocal a_buf, b_buf, c_buf, sa_buf, sb_buf, sc_buf
        if not a_buf:
            return
        writer.write_table(pa.table({
            "kmerA": a_buf, "kmerB": b_buf, "kmerC": c_buf,
            "sig_a": sa_buf, "sig_b": sb_buf, "sig_c": sc_buf,
        }, schema=_TRIP_SCHEMA))
        a_buf, b_buf, c_buf = [], [], []
        sa_buf, sb_buf, sc_buf = [], [], []

    for i, j in combinations(range(n_sig), 2):
        sij = pruned[i]["strain_set"] & pruned[j]["strain_set"]
        for k in range(j + 1, n_sig):
            if sij.isdisjoint(pruned[k]["strain_set"]):
                sid_i, sid_j, sid_k = pruned[i]["sig_id"], pruned[j]["sig_id"], pruned[k]["sig_id"]
                for ka in pruned[i]["kmers"]:
                    for kb in pruned[j]["kmers"]:
                        for kc in pruned[k]["kmers"]:
                            trip = sorted([(ka, sid_i), (kb, sid_j), (kc, sid_k)])
                            a_buf.append(trip[0][0]); b_buf.append(trip[1][0]); c_buf.append(trip[2][0])
                            sa_buf.append(trip[0][1]); sb_buf.append(trip[1][1]); sc_buf.append(trip[2][1])
                            n_inform += 1
                            if len(a_buf) >= batch_size:
                                flush()

    flush()
    writer.close()
    print(f"  Informative triplets: {n_inform:,}", flush=True)
    return n_inform

In [23]:
import os
import shutil
import polars as pl

# assume the functions above are in scope, otherwise import them

OUTPUT_DIR = "/tmp/sig_test"
BASENAME = "testmode"

# fresh output dir
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1. Build test data ────────────────────────────────────────────────────
df, df_samples, df_long = get_test_dataset()

# ── 2. Singletons (kmers absent from every effective strain) ──────────────
# In df_long this is exactly the kmers that don't appear at all.
all_kmers = set(df["#kmer"].to_list())
kmers_with_strain = set(df_long["#kmer"].to_list())
inform_singletons = sorted(all_kmers - kmers_with_strain)
print("\n=== INFORMATIVE SINGLETONS ===")
print(inform_singletons)
print(f"  expected: ['A1', 'A2']  →  got {len(inform_singletons)}")

# ── 3. Build signatures ───────────────────────────────────────────────────
print("\n=== SIGNATURE BUILD ===")
sig_df, kmer_to_sig, strain_index = build_signatures_from_long(df_long)
print("\nSignature table:")
print(sig_df)
print("\nStrain index:", strain_index)

# ── 4. Inspect signature size distribution ────────────────────────────────
print("\n=== SIGNATURE SIZE DISTRIBUTION ===")
print(sig_df.select(["sig_id", "n_strains", "n_kmers", "kmers"])
            .sort("n_kmers", descending=True))

# pairs
n_pairs = find_informative_pairs(sig_df, OUTPUT_DIR, BASENAME)
inform_pairs = pl.read_parquet(os.path.join(OUTPUT_DIR, f"{BASENAME}.inform_kmer_pairs.parquet"))
print(inform_pairs.sort(["kmerA", "kmerB"]))

# k-mers consumed by pairs
used_kmers = set(inform_pairs["kmerA"].to_list()) | set(inform_pairs["kmerB"].to_list())
print(f"  k-mers in pairs (excluded from triplet search): {len(used_kmers)}")

# triplets, restricted to leftover k-mers
n_trips = find_informative_triplets(sig_df, OUTPUT_DIR, BASENAME, used_kmers=used_kmers)
inform_triplets = pl.read_parquet(os.path.join(OUTPUT_DIR, f"{BASENAME}.inform_kmer_triplets.parquet"))
print(inform_triplets.sort(["kmerA", "kmerB", "kmerC"]))

Creating Test Data
Strain panel:
shape: (12, 8)
┌───────┬──────┬──────┬──────┬──────┬──────┬──────┬──────────┐
│ #kmer ┆ ST-1 ┆ ST-2 ┆ ST-3 ┆ ST-4 ┆ ST-5 ┆ ST-6 ┆ ST-empty │
│ ---   ┆ ---  ┆ ---  ┆ ---  ┆ ---  ┆ ---  ┆ ---  ┆ ---      │
│ str   ┆ i64  ┆ i64  ┆ i64  ┆ i64  ┆ i64  ┆ i64  ┆ i64      │
╞═══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════════╡
│ A1    ┆ 0    ┆ 0    ┆ 0    ┆ 0    ┆ 0    ┆ 0    ┆ 0        │
│ A2    ┆ 0    ┆ 0    ┆ 0    ┆ 0    ┆ 0    ┆ 0    ┆ 0        │
│ T1    ┆ 1    ┆ 1    ┆ 0    ┆ 1    ┆ 1    ┆ 0    ┆ 0        │
│ T2    ┆ 1    ┆ 0    ┆ 1    ┆ 1    ┆ 0    ┆ 1    ┆ 0        │
│ T3    ┆ 0    ┆ 1    ┆ 1    ┆ 0    ┆ 1    ┆ 1    ┆ 0        │
│ …     ┆ …    ┆ …    ┆ …    ┆ …    ┆ …    ┆ …    ┆ …        │
│ U3    ┆ 0    ┆ 1    ┆ 1    ┆ 1    ┆ 1    ┆ 0    ┆ 0        │
│ P1    ┆ 1    ┆ 1    ┆ 1    ┆ 0    ┆ 0    ┆ 0    ┆ 0        │
│ P2    ┆ 0    ┆ 0    ┆ 0    ┆ 1    ┆ 1    ┆ 1    ┆ 0        │
│ P3    ┆ 1    ┆ 0    ┆ 0    ┆ 1    ┆ 1    ┆ 0    ┆ 0        │
│ P4   